In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

In [2]:
train_prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context. 
Write a response that appropriately completes the request. 
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning. 
Please answer the following medical question. 

### Question:
{}

### Response:
<think>
{}
</think>
{}"""

In [3]:
model_dir = "Qwen/Qwen3-32B"
tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)

In [4]:
tokenizer.additional_special_tokens

['<|im_start|>',
 '<|im_end|>',
 '<|object_ref_start|>',
 '<|object_ref_end|>',
 '<|box_start|>',
 '<|box_end|>',
 '<|quad_start|>',
 '<|quad_end|>',
 '<|vision_start|>',
 '<|vision_end|>',
 '<|vision_pad|>',
 '<|image_pad|>',
 '<|video_pad|>']

In [5]:
tokenizer.eos_token

'<|im_end|>'

In [6]:
EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN

def formatting_prompts_func(examples):
    inputs = examples["Question"]
    complex_cots = examples["Complex_CoT"]
    outputs = examples["Response"]
    texts = []
    for question, cot, response in zip(inputs, complex_cots, outputs):
        # Append the EOS token to the response if it's not already there
        if not response.endswith(tokenizer.eos_token):
            response += tokenizer.eos_token
        text = train_prompt_style.format(question, cot, response)
        texts.append(text)
    return {"text": texts}

In [7]:
from datasets import load_dataset

In [8]:
dataset = load_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT",
    "en",
    split="train[0:2000]",
    trust_remote_code=True,
)

In [9]:
dataset

Dataset({
    features: ['Question', 'Complex_CoT', 'Response'],
    num_rows: 2000
})

In [10]:
dataset[0]

{'Question': 'Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?',
 'Complex_CoT': "Okay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?\n\nBut wait, there's more. The right lower leg is swollen and tender, which is like waving a big flag for deep vein thrombosis, especially after a long flight or sitting around a lot.\n\nSo, now I'm thinking, how could a clot in the leg end up causing issues like weakness or stroke symptoms?\n\nOh, right! There's this thing called a paradoxical embolism. It can happen if there's some kind of short circuit in the heart - like a hole that shouldn't be there.\n\nLet's put this together: if a blood clot from the leg somehow travels to the l

In [11]:
formatted_dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)

In [12]:
formatted_dataset

Dataset({
    features: ['Question', 'Complex_CoT', 'Response', 'text'],
    num_rows: 2000
})

In [13]:
formatted_dataset["text"][0]

"Below is an instruction that describes a task, paired with an input that provides further context. \nWrite a response that appropriately completes the request. \nBefore answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.\n\n### Instruction:\nYou are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning. \nPlease answer the following medical question. \n\n### Question:\nGiven the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?\n\n### Response:\n<think>\nOkay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?\n\nBut wait, there's more. The right lower leg i

In [ ]:
formatted_dataset

Dataset({
    features: ['Question', 'Complex_CoT', 'Response', 'text'],
    num_rows: 2000
})

In [35]:
formatted_dataset.split

NamedSplit('train[0:2000]')

In [41]:
next(iter(formatted_dataset)).keys()

dict_keys(['Question', 'Complex_CoT', 'Response', 'text'])

In [ ]:
list(next(iter(dataset)).keys())

In [ ]:
from huggingface_hub import login, create_repo
from datasets import DatasetDict, Dataset

hf_token = ""
login(token=hf_token)

create_repo("ty-kim/medical_cot", repo_type="dataset", exist_ok=True)

dataset_to_push = DatasetDict({"train": formatted_dataset})

dataset_to_push.push_to_hub("ty-kim/medical_cot",
                    private=True,            # 공개로 올리려면 False
                    max_shard_size="1GB",    # → 파일 하나가 1 GB 넘으면 자동으로 shard
                    token=True,               # login() 대신 토큰 문자열 직접 넣어도 됨
                    commit_message="first",
                    ) # max_shard_size="500MB"

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/ty-kim/medical_cot/commit/97bd16a7eacae8eee2c6a0b941e1dbeaf0e970f2', commit_message='first', commit_description='', oid='97bd16a7eacae8eee2c6a0b941e1dbeaf0e970f2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/ty-kim/medical_cot', endpoint='https://huggingface.co', repo_type='dataset', repo_id='ty-kim/medical_cot'), pr_revision=None, pr_num=None)

In [42]:
my_dataset = load_dataset(
        "ty-kim/medical_cot",
        trust_remote_code=True,
    )

In [47]:
next(iter(my_dataset['train'])).keys()

dict_keys(['Question', 'Complex_CoT', 'Response', 'text'])

In [43]:
next(iter(my_dataset)).keys()

AttributeError: 'str' object has no attribute 'keys'

### Before Fine-tuning

In [14]:
# 학습전 추론 성능 확인
inference_prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context. 
Write a response that appropriately completes the request. 
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning. 
Please answer the following medical question. 

### Question:
{}

### Response:
<think>

"""

In [15]:
# 학습전 추론 성능 확인
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_dir,
    quantization_config=bnb_config,   
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True             
)

model.config.use_cache = False
model.config.pretraining_tp = 1

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

model-00003-of-00017.safetensors:   0%|          | 0.00/3.90G [00:00<?, ?B/s]

model-00001-of-00017.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00002-of-00017.safetensors:   0%|          | 0.00/3.90G [00:00<?, ?B/s]

model-00004-of-00017.safetensors:   0%|          | 0.00/3.90G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/17 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [18]:
tokenizer.special_tokens_map

{'eos_token': '<|im_end|>',
 'pad_token': '<|endoftext|>',
 'additional_special_tokens': ['<|im_start|>',
  '<|im_end|>',
  '<|object_ref_start|>',
  '<|object_ref_end|>',
  '<|box_start|>',
  '<|box_end|>',
  '<|quad_start|>',
  '<|quad_end|>',
  '<|vision_start|>',
  '<|vision_end|>',
  '<|vision_pad|>',
  '<|image_pad|>',
  '<|video_pad|>']}

In [ ]:
tokenizer.eos_token

'<|im_end|>'

In [19]:
EOS_TOKEN

'<|im_end|>'

In [ ]:
question = dataset[10]['Question']
inputs = tokenizer(
    [inference_prompt_style.format(question) + EOS_TOKEN],
    return_tensors="pt",
).to("cuda")

In [30]:
question

'In a patient with dermatomyositis as indicated by fatigue, muscle weakness, a scaly rash, elevated creatine kinase-MB, anti-Jo-1 antibodies, and perimysial inflammation, which type of cancer is most often associated with this condition?'

In [31]:
inputs

{'input_ids': tensor([[ 38214,    374,    458,   7600,    429,  16555,    264,   3383,     11,
          34426,    448,    458,   1946,    429,   5707,   4623,   2266,     13,
            715,   7985,    264,   2033,    429,  34901,  44595,    279,   1681,
             13,    715,  10227,  35764,     11,   1744,  15516,    911,    279,
           3405,    323,   1855,    264,   3019,  14319,  29208,   8781,    315,
          11303,    311,   5978,    264,  19819,    323,  13382,   2033,    382,
          14374,  29051,    510,   2610,    525,    264,   6457,   6203,    448,
          10847,   6540,    304,  14490,  32711,     11,  49418,     11,    323,
           6380,   9115,     13,    715,   5501,   4226,    279,   2701,   6457,
           3405,     13,   4710,  14374,  15846,    510,    641,    264,   8720,
            448,  60385,   5533,    436,  19435,    438,  16317,    553,  35609,
             11,  15747,  23078,     11,    264,   1136,   5774,  56242,     11,
          3128

In [32]:
tokenizer.decode(inputs.input_ids[0], skip_special_tokens=True)

'Below is an instruction that describes a task, paired with an input that provides further context. \nWrite a response that appropriately completes the request. \nBefore answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.\n\n### Instruction:\nYou are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning. \nPlease answer the following medical question. \n\n### Question:\nIn a patient with dermatomyositis as indicated by fatigue, muscle weakness, a scaly rash, elevated creatine kinase-MB, anti-Jo-1 antibodies, and perimysial inflammation, which type of cancer is most often associated with this condition?\n\n### Response:\n<think>\n\n'

In [25]:
outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    eos_token_id=tokenizer.eos_token_id,
    use_cache=True
)

In [26]:
outputs

tensor([[ 38214,    374,    458,   7600,    429,  16555,    264,   3383,     11,
          34426,    448,    458,   1946,    429,   5707,   4623,   2266,     13,
            715,   7985,    264,   2033,    429,  34901,  44595,    279,   1681,
             13,    715,  10227,  35764,     11,   1744,  15516,    911,    279,
           3405,    323,   1855,    264,   3019,  14319,  29208,   8781,    315,
          11303,    311,   5978,    264,  19819,    323,  13382,   2033,    382,
          14374,  29051,    510,   2610,    525,    264,   6457,   6203,    448,
          10847,   6540,    304,  14490,  32711,     11,  49418,     11,    323,
           6380,   9115,     13,    715,   5501,   4226,    279,   2701,   6457,
           3405,     13,   4710,  14374,  15846,    510,    641,    264,   8720,
            448,  60385,   5533,    436,  19435,    438,  16317,    553,  35609,
             11,  15747,  23078,     11,    264,   1136,   5774,  56242,     11,
          31289,   6056,    

In [ ]:
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)

In [28]:
print(response[0])

Below is an instruction that describes a task, paired with an input that provides further context. 
Write a response that appropriately completes the request. 
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning. 
Please answer the following medical question. 

### Question:
In a patient with dermatomyositis as indicated by fatigue, muscle weakness, a scaly rash, elevated creatine kinase-MB, anti-Jo-1 antibodies, and perimysial inflammation, which type of cancer is most often associated with this condition?

### Response:
<think>



The patient described has dermatomyositis, characterized by fatigue, muscle weakness, a scaly rash, elevated creatine kinase-MB (which is more specific for cardiac muscle but sometimes elevated in myopathies), anti-Jo-1 antibodies (suggestive of a

In [ ]:
# </think>로 닫힌 뒤 최종 Respose에 답을 내놓는 형식과는 다름. 그래도 뭔가 하려고 하는듯